# Preserved mesh repair
No resampling or paid calls. Source volume and compiled dependencies are checked by hash.

In [ ]:
SOURCE_HASH='761c6520b1d38cef68e4fdd3f4898916e49648bd8413e2c3a79abd4ade5ac6b2'
EXPECTED={'cumesh-0.0.1-cp313-cp313-linux_x86_64.whl': 'a60c661c67d19b96648c98c8d7a6bb80bcd190b446fac4afe01e9b5b0b01650e', 'nvdiffrast-0.4.0-cp313-cp313-linux_x86_64.whl': '992d930c68af3c40f9261d81adbd10dc53b1ee7dd5e97b67b3762f309c4be3c0', 'o_voxel-0.0.1-cp313-cp313-linux_x86_64.whl': '10f987ebce9c78c47a269b6321a00a1ad05faa312b31b8151a19b92c7b9fa38c', 'flex_gemm-1.0.0-cp313-cp313-linux_x86_64.whl': '07b57660b62a728c8e44a7902127127a8e28f2b548cfd274fd3f3f6a3573527c'}

import pathlib,os,json,hashlib,time,subprocess,sys,traceback
OUT=pathlib.Path('/kaggle/working/dragon-mesh-repair');OUT.mkdir(parents=True,exist_ok=True)
TMP=pathlib.Path('/kaggle/temp/dragon-mesh-repair');TMP.mkdir(parents=True,exist_ok=True)
os.environ.update(TMPDIR=str(TMP),PIP_CACHE_DIR=str(TMP/'pip'),TORCH_CUDA_ARCH_LIST='7.5')
started=time.monotonic();report={'stage':'setup','art_accepted':False,'geometry_resampled':False,'source_npz_sha256':SOURCE_HASH}
def save():
    report['elapsed_seconds']=round(time.monotonic()-started,2)
    (OUT/'repair.json').write_text(json.dumps(report,indent=2));print(json.dumps(report),flush=True)
def run(args,name,timeout=180):
    with (OUT/(name+'.log')).open('w') as f:subprocess.run(args,stdout=f,stderr=subprocess.STDOUT,check=True,timeout=timeout)
save()
try:
    run([sys.executable,'-m','pip','install','trimesh','opencv-python-headless','pillow','scipy','plyfile','tqdm','imageio','zstandard'],'dependencies',240)
    run([sys.executable,'-m','pip','install','git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8'],'utils3d')
    wheels=[]
    for name,sha in EXPECTED.items():
        p=next(pathlib.Path('/kaggle/input').rglob(name))
        if hashlib.sha256(p.read_bytes()).hexdigest()!=sha:raise RuntimeError('Wheel hash mismatch')
        wheels.append(str(p))
    run([sys.executable,'-m','pip','install','--no-deps',*wheels],'wheels',90)
    repo=TMP/'TRELLIS.2'
    run(['git','clone','--filter=blob:none','https://github.com/microsoft/TRELLIS.2.git',str(repo)],'clone',120)
    run(['git','-C',str(repo),'checkout','75fbf0183001ed9876c8dbb35de6b68552ee08bd'],'checkout',60)
    sys.path.insert(0,str(repo));import torch,numpy as np,o_voxel.postprocess
    source=next(pathlib.Path('/kaggle/input').rglob('source-volume.npz'))
    if hashlib.sha256(source.read_bytes()).hexdigest()!=SOURCE_HASH:raise RuntimeError('Source artifact mismatch')
    layout=json.loads((source.parent/'source-volume-layout.json').read_text())['layout']
    with np.load(source,allow_pickle=False) as z:
        verts=torch.from_numpy(z['vertices']).cuda();faces=torch.from_numpy(z['faces']).cuda()
        attrs=torch.from_numpy(z['attrs']).cuda();coords=torch.from_numpy(z['coords']).cuda()
        voxel=float(z['voxel_size'])
    report.update(stage='repairing',remesh=True,remesh_band=1,remesh_project=0,decimation_target=1000000,texture_size=2048);save()
    with torch.no_grad():
        glb=o_voxel.postprocess.to_glb(vertices=verts,faces=faces,attr_volume=attrs,coords=coords,attr_layout={k:slice(*v) for k,v in layout.items()},voxel_size=voxel,aabb=[[-.5,-.5,-.5],[.5,.5,.5]],decimation_target=1000000,texture_size=2048,remesh=True,remesh_band=1,remesh_project=0,verbose=True)
    path=OUT/'hostile-dragon-repaired.glb';glb.export(path,extension_webp=True)
    report.update(stage='repaired_candidate_saved',bytes=path.stat().st_size,sha256=hashlib.sha256(path.read_bytes()).hexdigest(),vertices=len(glb.vertices),faces=len(glb.faces));save()
except Exception as e:
    report.update(stage='repair_failed',error=repr(e),traceback=traceback.format_exc());save()
